In [ ]:
# KD-ReID on Kaggle: evaluate trained runs attached as inputs (plain + per-domain BN statistics).
# Inputs: the dataset with the dataset zips, plus either the output of the notebook that trained the
# run (folder <name>/ with last.pth + config.yaml) or files <name>_last.pth + <name>_config.yaml.
# Settings: GPU T4 x2, Internet On, secret GH_TOKEN. Keep this as the ONLY cell.
NAMES = ["6c_dkd_clip_x1_s1", "8a_unl_market", "8b_unl_cuhk03", "8b_unl_cuhk03_s1"]

import os, shutil, subprocess, zipfile
from pathlib import Path

IN, TMP = Path("/kaggle/input"), Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
RAW, DATA, WORK, REPO = TMP / "raw", TMP / "data", Path("/kaggle/working"), TMP / "KD-ReID"

def sh(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode

import torch
assert torch.cuda.is_available(), "Settings > Accelerator > GPU T4 x2"
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
if not REPO.exists():
    assert sh(f"git clone -q https://{token}@github.com/cts0424/KD-ReID.git {REPO}") == 0
sh(f"git -C {REPO} pull -q --ff-only && git -C {REPO} log --oneline -1")
assert sh(f"pip install -q -e {REPO}") == 0
os.chdir(REPO)

for z in IN.rglob("*.zip"):
    if not (RAW / z.stem).exists():
        zipfile.ZipFile(z).extractall(RAW / z.stem)
def find_split_dir(*kw):
    hits = [p.parent for root in (IN, RAW) if root.exists() for p in root.rglob("bounding_box_train")
            if "__macosx" not in str(p).lower() and all(k in str(p).lower() for k in kw)]
    assert hits, f"no bounding_box_train with {kw}"
    return sorted(hits, key=lambda p: len(str(p)))[0]
for dst, src in {DATA / "Market-1501-v15.09.15": find_split_dir("market"),
                 DATA / "cuhk03-np" / "detected": find_split_dir("detected")}.items():
    if not dst.exists():
        shutil.copytree(src, dst)

runs = []
for name in NAMES:
    dirs = [p.parent for p in IN.rglob("last.pth") if p.parent.name == name and (p.parent / "config.yaml").exists()]
    if dirs:
        runs.append(str(dirs[0])); continue
    ck, cf = list(IN.rglob(f"{name}_last.pth")), list(IN.rglob(f"{name}_config.yaml"))
    if ck and cf:
        d = TMP / "runs" / name; d.mkdir(parents=True, exist_ok=True)
        shutil.copy(ck[0], d / "last.pth"); shutil.copy(cf[0], d / "config.yaml")
        runs.append(str(d)); continue
    print(f"!! {name} not found in the inputs: skipped")
assert runs, "no run found: attach the training notebook output (Add Input > Notebooks)"
print("evaluating:", runs)
sh(f"python tools/eval_matrix.py --runs {' '.join(runs)} --bn-recal "
   f"--datasets market1501:{DATA} cuhk03np:{DATA} --out {WORK}/eval_matrix_adabn.json")
